# Excel extraction lab — pandas vs openpyxl

`excelExtractText` can read Excel in two ways, chosen with `method`:

* **`"pandas"`** (the default, the original) — one markdown table per sheet, generated column numbers `0 1 2 …` on top.
* **`"openpyxl"`** — reads cell by cell and keeps **Excel's own column letters and row numbers**, so a model can cite `'Q3 Report'!B7`.

This notebook runs both on the same messy input, **`samples/excel_edge_cases.xlsx`** (14 tabs, each a different kind of mess), and shows what each one gives the LLM. The text printed below is exactly what the LLM would receive.

Full reference: [`docs/excel_to_markdown.md`](../docs/excel_to_markdown.md).

## 1. Setup

In [1]:
import sys, json, tempfile
from pathlib import Path
import openpyxl
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import local_harness as h            # imports src/app.py and dispatches by function name
from make_samples import build_all

S = build_all(ROOT / "samples")      # (re)generate the sample documents; the edge-case workbook is copied along
BOOK = S["excel_edge_cases.xlsx"]
print("workbook:", BOOK.relative_to(ROOT))


def sheets(method, **options):
    """{tab name: text} for the whole workbook, using one method."""
    out = h.run("excelExtractText", BOOK, method=method, **options)
    return {s["sheet_name"]: s["text"] for s in out}


PANDAS = sheets("pandas")
PLAIN = sheets("openpyxl", include_legend=False)    # openpyxl, defaults; legend off so the cells below stay short


def compare(tab, limit=2600, **options):
    """Print one tab as the pandas method sees it, then as the openpyxl method sees it."""
    other = sheets("openpyxl", include_legend=False, **options) if options else PLAIN
    print(f"=============== {tab!r}: pandas method ===============")
    print(PANDAS[tab][:limit] + (" …" if len(PANDAS[tab]) > limit else ""))
    print(f"\n=============== {tab!r}: openpyxl method {options or ''} ===============")
    print(other[tab][:limit] + (" …" if len(other[tab]) > limit else ""))

workbook: samples/excel_edge_cases.xlsx


## 2. The workbook

Fourteen tabs. Two are hidden on purpose: one **hidden**, one **very hidden** (the kind Excel's *Unhide* menu cannot show).

In [2]:
wb = openpyxl.load_workbook(BOOK)
for n, ws in enumerate(wb.worksheets, 1):
    print(f"{n:2}  {ws.title:18} {ws.sheet_state}")

 1  Summary            visible
 2  Q3 Report          visible
 3  Multi Tables       visible
 4  No Header Data     visible
 5  Hidden Rates       hidden
 6  Config Secret      veryHidden
 7  Hidden Rows Cols   visible
 8  Empty              visible
 9  Big Log            visible
10  Messy              visible
11  Merged Cells       visible
12  Comments           visible
13  Dates              visible
14  General Dates      visible


## 3. Where is the header? (read this first)

In the markdown tables the **top row is not your header**. It is only Excel's column letters (`A B C`), or `0 1 2` with the pandas method. Your table's own header (`Product`, `Units`, `Price`) is an ordinary cell of the sheet, so it shows up as the **first data row**, right under the letters. The date detection in section 9 depends on this, so it is worth seeing once.

In [3]:
compare("Summary")

=============== 'Summary': pandas method ===============
| 0 | 1 | 2 |
|:----------|:------|:------|
| Product | Units | Price |
| Widget | 120 | 9.99 |
| Gadget | 75 | 24.5 |
| Gizmo | 30 | 0.3 |
| Doohickey | 200 | 4 |

=============== 'Summary': openpyxl method  ===============
## Sheet Summary — visible

### Table 1 — Summary!A1:C5

| | A | B | C |
|---|---|---|---|
| 1 | Product | Units | Price |
| 2 | Widget | 120 | 9.99 |
| 3 | Gadget | 75 | 24.5 |
| 4 | Gizmo | 30 | 0.3 |
| 5 | Doohickey | 200 | 4 |



## 4. Several tables on one sheet, notes and key-value blocks

`Multi Tables` holds a headcount table, a budget table **beside** it, a table with **no header**, a stray note far away, and a small key-value block. The pandas method puts everything into one grid, so the budget table shares rows with the headcount table. The openpyxl method finds each block and gives it its cell range.

In [4]:
compare("Multi Tables")

=============== 'Multi Tables': pandas method ===============
| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|:----|:------------------------|:--------------------|:-----------|:----|:----|:----|:------|:--------|:----|:---------------|
| | | | | | | | | | | |
| | Headcount by department | | | | | | | | | |
| | Dept | Heads | Open roles | | | | Dept | Budget | | |
| | Eng | 40 | 5 | | | | Eng | 4000000 | | |
| | Sales | 25 | 2 | | | | Sales | 1500000 | | |
| | Ops | 12 | 0 | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | 1 | 2 | 3 | | | | | | | |
| | 4 | 5 | 6 | | | | | | | |
| | 7 | 8 | 9 | | | | | | | |
| | 10 | 11 | 12 | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | See appendix B |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | Owner | J. Rivera | | | | | 

`Q3 Report` has a title and notes above the table, a two-level header, formulas (their saved results are shown), and a footnote.

In [5]:
compare("Q3 Report")

=============== 'Q3 Report': pandas method ===============
| 0 | 1 | 2 | 3 | 4 |
|:---------------------------------------------------------|:--------------------|:-----|:-----|:------|
| Q3 Regional Sales Report | | | | |
| Prepared by: Finance Team | | | | |
| Date: | 2026-10-01 00:00:00 | | | |
| Region | Q3 Revenue ($) | | | Total |
| | Jul | Aug | Sep | |
| East | 1200 | 1350 | 1100 | 3650 |
| West | 900 | 950 | 1010 | 2860 |
| North | 400 | 420 | 380 | 1200 |
| South | 760 | 800 | 845 | 2405 |
| Total | 3260 | 3520 | 3335 | 10115 |
| Note: figures are unaudited \| source: ERP export. | | | | |
| Second line of the note. | | | | |
| Footnote spanning the table width, merged across A to E. | | | | |

=============== 'Q3 Report': openpyxl method  ===============
## Sheet 'Q3 Report' — visible

Regions on this sheet: table A1:E3, table A5:E11, table A13:E14

### Table 1 — 'Q3 Report'!A1:E3

| | A | B |
|---|---|---|
| 1 | Q3 Regional Sales Report |  |
| 2 | Prepared by: Finance Team 

## 5. Merged cells: horizontal, vertical and combined

The value sits in the first cell of a merged range; the other cells are blank. With pandas nothing says a cell was merged. With openpyxl the merged ranges are listed under the table, so "Europe" is known to cover the Spain and Italy rows, and "H1" covers both Q1 and Q2.

In [6]:
compare("Merged Cells")

=============== 'Merged Cells': pandas method ===============
| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|:------------------------------|:--------|:----|:----|:----|:----|:----|:------------|:------|:----|:-----|
| Sales by Region and Half-year | | | | | | | | | | |
| | | | | | | | | | | |
| Region | Country | H1 | | H2 | | | Target band | | Low | High |
| | | Q1 | Q2 | Q3 | Q4 | | | | 1 | 2 |
| Europe | France | 10 | 12 | 14 | 16 | | Europe | Total | 3 | 4 |
| | Spain | 8 | 9 | 11 | 13 | | | | | |
| | Italy | 7 | 7 | 9 | 10 | | | | | |
| Asia | Japan | 20 | 22 | 21 | 25 | | | | | |
| | India | 15 | 18 | 19 | 24 | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| | | | | | | | | | | |
| After the empty merge | | | | | | | | | | |

=============== 'Merged Cells': openpyxl method  ===============
## Sheet 'Merged Cells' — visible

Regions on this sheet: cell A1:F1, table A3:F9, table H3:K5, cell A13

**A1** (merged cell A1:F1): Sales by Region and Half-year

- Merged cells

## 6. Hidden sheets, rows and columns

Both methods read hidden sheets. The openpyxl method **labels** them (and marks hidden rows and columns); you can also leave them out.

In [7]:
print("pandas reads:      ", list(PANDAS))
print("openpyxl headings: ", [ln for t in PLAIN.values() for ln in t.splitlines() if ln.startswith("## Sheet") and "HIDDEN" in ln])
print()
without = sheets("openpyxl", include_hidden_sheets=False)
print("with include_hidden_sheets=False:", list(without))
print()
compare("Hidden Rows Cols")

pandas reads:       ['Summary', 'Q3 Report', 'Multi Tables', 'No Header Data', 'Hidden Rates', 'Config Secret', 'Hidden Rows Cols', 'Empty', 'Big Log', 'Messy', 'Merged Cells', 'Comments', 'Dates', 'General Dates']
openpyxl headings:  ["## Sheet 'Hidden Rates' — HIDDEN SHEET", "## Sheet 'Config Secret' — VERY HIDDEN SHEET"]

with include_hidden_sheets=False: ['Summary', 'Q3 Report', 'Multi Tables', 'No Header Data', 'Hidden Rows Cols', 'Empty', 'Big Log', 'Messy', 'Merged Cells', 'Comments', 'Dates', 'General Dates']

=============== 'Hidden Rows Cols': pandas method ===============
| 0 | 1 | 2 | 3 |
|:----|:-----|:------------|:-------|
| ID | Name | Salary band | Status |
| 1 | Ana | B2 | Active |
| 2 | Ben | B3 | Left |
| 3 | Cy | B1 | Active |
| 4 | Dee | B4 | Left |

=============== 'Hidden Rows Cols': openpyxl method  ===============
## Sheet 'Hidden Rows Cols' — visible

Hidden rows: [3, 5]; hidden columns: ['C'] (shown below and marked)

### Table 1 — 'Hidden Rows Cols'!A1:D5



## 7. Comments (opt-in)

Comments are **off by default**. With `include_comments=True` they are listed under their table, including comments on empty cells and on cells outside any table (the last block of this sheet), and Excel's "threaded comment" boilerplate is removed.

In [8]:
compare("Comments", include_comments=True)

=============== 'Comments': pandas method ===============
| 0 | 1 | 2 |
|:-----------------------|:-----|:---------|
| Item | Cost | Approved |
| Pen | 2.5 | Yes |
| Desk | 180 | |
| Lamp | 35 | Yes |
| | | |
| | | |
| Legend: Yes = approved | | |

=============== 'Comments': openpyxl method {'include_comments': True} ===============
## Sheet Comments — visible

Regions on this sheet: table A1:C4, cell A7, comments on E10

### Table 1 — Comments!A1:C4

| | A | B | C |
|---|---|---|---|
| 1 | Item | Cost | Approved |
| 2 | Pen | 2.5 | Yes |
| 3 | Desk | 180 |  |
| 4 | Lamp | 35 | Yes |

- Comment on B1: Cost in USD, excluding tax.
- Comment on B3: Quote expired. / Ask the vendor for a new one.
- Comment on C3: Waiting for sign-off
- Comment on A4: Comment: Check this figure with Ops / Reply: Confirmed, figure is correct

**A7** (single cell): Legend: Yes = approved

- Comment on A7: Added by audit

### Comments on cells outside any table — Comments

- Comment on E10: Orphan note: ignore

## 8. Dates that Excel has formatted as dates

Excel stores a date as a plain number plus a display format. Both methods turn it back into a real date, but the openpyxl method writes it the way Excel shows it: no `00:00:00` on plain dates, elapsed hours as `36:30:00` (not `1 day, 12:30:00`), and a zero date as `1900-01-00` (not `00:00:00`, which looks like midnight). The right-hand column of the sheet says what Excel displays.

In [9]:
compare("Dates")

=============== 'Dates': pandas method ===============
| 0 | 1 | 2 |
|:---------------------------|:--------------------|:-----------------|
| Case | Stored value | Excel displays |
| Date (dd/mm/yyyy) | 2026-10-05 00:00:00 | 05/10/2026 |
| Date (month and year only) | 2026-10-05 00:00:00 | Oct 2026 |
| Date and time | 2026-10-05 14:30:00 | 2026-10-05 14:30 |
| Time of day | 09:15:30 | 09:15:30 |
| Elapsed time (hours) | 1 day, 12:30:00 | 36:30 |
| Date from a formula | 2026-11-04 00:00:00 | 04/11/2026 |
| Empty date (zero) | 00:00:00 | 00/01/1900 |
| Date stored as text | 2026-10-05 | 2026-10-05 |
| Number, not a date | 46300 | 46300 |

=============== 'Dates': openpyxl method  ===============
## Sheet Dates — visible

### Table 1 — Dates!A1:C10

| | A | B | C |
|---|---|---|---|
| 1 | Case | Stored value | Excel displays |
| 2 | Date (dd/mm/yyyy) | 2026-10-05 | 05/10/2026 |
| 3 | Date (month and year only) | 2026-10-05 | Oct 2026 |
| 4 | Date and time | 2026-10-05 14:30:00 | 2026-10-

## 9. Dates stored as plain numbers (General format) — opt-in

A date in a **General**-format cell is just a number (`46300`); Excel shows `46300` too, so by default both methods write it as is. A model may not recognise `46300` as a date. `detect_general_dates=True` (openpyxl method only) shows the date **and keeps the original number**: `2026-10-05 (serial 46300)`. It is cautious: it only converts under a **date-like header**, and only if the number is in a plausible range.

### What "column header" means

The header is a **cell of the sheet**, not the letters row. For each column the converter takes (1) the unbroken stack of text cells directly above the first General-format number, and (2) a merged cell directly above that stack. First, the proof that the header is an ordinary cell: here are the first rows of the sheet itself.

In [10]:
ws = openpyxl.load_workbook(BOOK, data_only=True)["General Dates"]
for r in range(1, 5):
    print(r, [c.value for c in ws[r]][:7])

1 ['Order ID', 'Due Date', 'Created', 'Amount Due', 'Date Code', 'Updated', 'Posted']
2 [46300, 46300, 46300.6041666667, 46300, 12, 46300, datetime.datetime(2026, 10, 5, 0, 0)]
3 [46301, 46330, 46301.5, 46301, 15, 'n/a', datetime.datetime(2026, 11, 4, 0, 0)]
4 [46302, 46361, 46302.25, 46302, 7, 46303, datetime.datetime(2026, 12, 5, 0, 0)]


Now the same rows in the markdown. Row 1 is the header, and it is just the first row under `A B C`. Detection is **off** (the default), so the numbers stay numbers in both methods.

In [11]:
compare("General Dates", limit=1100)

=============== 'General Dates': pandas method ===============
| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|:---------|:---------|:---------|:-----------|:----------|:--------|:--------------------|:----|:---------------|:------|
| Order ID | Due Date | Created | Amount Due | Date Code | Updated | Posted | | Contract dates | |
| 46300 | 46300 | 46300.6 | 46300 | 12 | 46300 | 2026-10-05 00:00:00 | | Start | End |
| 46301 | 46330 | 46301.5 | 46301 | 15 | | 2026-11-04 00:00:00 | | 46300 | 46665 |
| 46302 | 46361 | 46302.25 | 46302 | 7 | 46303 | 2026-12-05 00:00:00 | | 46301 | 46666 |
| | | | | | | | | | |
| | | | | | | | | | |
| 46300 | 46301 | | | | | | | | |
| 46302 | 46303 | | | | | | | | |
| | | | | | | | | | |
| | | | | | | | | | |
| Claim | Due Date | | | Claim | Dates | | | | |
| A | TBD | | | | Start | | | | |
| B | 46300 | | | B | 46300 | | | | |
| C | 46301 | | | C | 46301 | | | | |

=============== 'General Dates': openpyxl method  ===============
## Sheet 'General Dates' — visib

With detection **on**: `Due Date`, `Created` and `Updated` are converted. `Order ID` and `Amount Due` hold the same numbers and are left alone (their headers do not look like dates, and "Amount Due" also contains an amount word). `Date Code` has a date-like name but its numbers (12, 15, 7) are not date-sized. `Posted` already has a real date format. The note under the table says which columns were converted.

In [12]:
on = sheets("openpyxl", include_legend=False, detect_general_dates=True)["General Dates"]
print(on[:on.index("### Table 2")])

## Sheet 'General Dates' — visible

Regions on this sheet: table A1:G4, table I1:J4, table A7:B8, table A11:B14, table E11:F14

### Table 1 — 'General Dates'!A1:G4

| | A | B | C | D | E | F | G |
|---|---|---|---|---|---|---|---|
| 1 | Order ID | Due Date | Created | Amount Due | Date Code | Updated | Posted |
| 2 | 46300 | 2026-10-05 (serial 46300) | 2026-10-05 14:30:00 (serial 46300.6041666667) | 46300 | 12 | 2026-10-05 (serial 46300) | 2026-10-05 |
| 3 | 46301 | 2026-11-04 (serial 46330) | 2026-10-06 12:00:00 (serial 46301.5) | 46301 | 15 | n/a | 2026-11-04 |
| 4 | 46302 | 2026-12-05 (serial 46361) | 2026-10-07 06:00:00 (serial 46302.25) | 46302 | 7 | 2026-10-08 (serial 46303) | 2026-12-05 |

- Date detection (opt-in): column B ("Due Date"): 3 number(s) stored in General format are shown as dates, with the original number in brackets
- Date detection (opt-in): column C ("Created"): 3 number(s) stored in General format are shown as dates, with the original number in brackets
- Date 

**The header can be more than one row.** In the first table below a text value, `TBD`, sits between the header and the first number. In the second, there are two header rows that are *not* merged (`Dates` above `Start`). In the third, a *merged* group header ("Contract dates") covers `Start` and `End`. All three work. The last table has numbers and **no header**, so it is never touched.

In [13]:
for title in ("### Table 4", "### Table 5", "### Table 2", "### Table 3"):
    start = on.index(title)
    end = on.find("\n### ", start + 5)
    print(on[start: end if end != -1 else None].rstrip(), "\n")

### Table 4 — 'General Dates'!A11:B14

| | A | B |
|---|---|---|
| 11 | Claim | Due Date |
| 12 | A | TBD |
| 13 | B | 2026-10-05 (serial 46300) |
| 14 | C | 2026-10-06 (serial 46301) |

- Date detection (opt-in): column B ("Due Date"): 2 number(s) stored in General format are shown as dates, with the original number in brackets 

### Table 5 — 'General Dates'!E11:F14

| | E | F |
|---|---|---|
| 11 | Claim | Dates |
| 12 |  | Start |
| 13 | B | 2026-10-05 (serial 46300) |
| 14 | C | 2026-10-06 (serial 46301) |

- Date detection (opt-in): column F ("Dates"): 2 number(s) stored in General format are shown as dates, with the original number in brackets 

### Table 2 — 'General Dates'!I1:J4

| | I | J |
|---|---|---|
| 1 | Contract dates |  |
| 2 | Start | End |
| 3 | 2026-10-05 (serial 46300) | 2027-10-05 (serial 46665) |
| 4 | 2026-10-06 (serial 46301) | 2027-10-06 (serial 46666) |

- Merged cells (value sits in the first cell of the range): I1:J1 ("Contract dates")
- Date detection (op

**Your own header words.** `Start` is not in the built-in list, so a column headed `Start` is left alone until you name it with `date_keywords` (this replaces the built-in list).

In [14]:
tmp = Path(tempfile.mkdtemp()) / "start_dates.xlsx"
wb2 = openpyxl.Workbook(); ws2 = wb2.active; ws2.title = "Contracts"
ws2.append(["Ref", "Start"]); ws2.append(["A", 46300]); ws2.append(["B", 46301]); wb2.save(tmp)

def row2(**options):
    text = h.run("excelExtractText", tmp, method="openpyxl", include_legend=False, **options)[0]["text"]
    return [ln for ln in text.splitlines() if ln.startswith("| 2 ")][0]

print("default                         :", row2(detect_general_dates=True))
print('date_keywords=["start"]         :', row2(detect_general_dates=True, date_keywords=["start"]))

default                         : | 2 | A | 46300 |
date_keywords=["start"]         : | 2 | A | 2026-10-05 (serial 46300) |


## 10. Long tables

`Big Log` has 450 rows. The openpyxl method splits it into parts of 200 rows, repeats the header row in each part, and keeps the real row numbers, so citations stay correct.

In [15]:
print("pandas: one table,", PANDAS["Big Log"].count(chr(10)) + 1, "lines")
print("openpyxl part titles:")
for line in PLAIN["Big Log"].splitlines():
    if line.startswith("### "):
        print("  ", line)
print()
part2 = PLAIN["Big Log"].split("part 2 of 3")[1].splitlines()
print("\n".join(part2[:7]))

pandas: one table, 453 lines
openpyxl part titles:
   ### Table 1 — 'Big Log'!A1:D451 (part 1 of 3, rows 1–200)
   ### Table 1 — 'Big Log'!A1:D451 (part 2 of 3, rows 201–400)
   ### Table 1 — 'Big Log'!A1:D451 (part 3 of 3, rows 401–451)

, rows 201–400)

| | A | B | C | D |
|---|---|---|---|---|
| 1 (first row repeated for context) | Row ID | Event | Value | Flag |
| 201 | 200 | event_200 | 300 | FALSE |
| 202 | 201 | event_201 | 301.5 | FALSE |


## 11. Messy and empty tabs

`Messy` has cells scattered across the sheet (A1, F2, D10, J30 … Z50), a very long cell, a pipe character and a line break inside cells, an error value, and one big merged block. The pandas method has to print a grid that spans the whole sheet; the openpyxl method lists each block.

In [16]:
print("pandas text length   :", len(PANDAS["Messy"]), "characters")
print("openpyxl text length :", len(PLAIN["Messy"]), "characters\n")
print(PLAIN["Messy"][:1800])
print("…\n")
print("Empty tab ->", repr(PLAIN["Empty"].splitlines()[-1]))

pandas text length   : 3883 characters
openpyxl text length : 1580 characters

## Sheet Messy — visible

Regions on this sheet: cell A1, cell F2, cell M5:R12, cell H8, cell D10, table J30:K31, cell Z50

**A1** (single cell): Top-left note

**F2** (single cell): xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

## 12. File types and fallback

openpyxl can only read `.xlsx` / `.xlsm`. Legacy `.xls` files are handed to pandas automatically (or raise an error with `fallback_to_pandas=False`). A real `.xlsx` that has a `.xls` name is still read by openpyxl, because the file's contents decide, not its name.

In [17]:
legacy = h.run("excelExtractText", S["legacy_register.xls"], method="openpyxl")
print("legacy .xls with method='openpyxl' -> first line:", legacy[0]["text"].splitlines()[0][:50], " (pandas was used)")
try:
    h.run("excelExtractText", S["legacy_register.xls"], method="openpyxl", fallback_to_pandas=False)
except Exception as e:
    print("with fallback_to_pandas=False     ->", type(e).__name__ + ":", e)
misnamed = h.run("excelExtractText", S["loss_run_misnamed.xls"], method="openpyxl")
print("misnamed .xlsx (named .xls)       -> first table title:", [ln for ln in misnamed[0]["text"].splitlines() if ln.startswith("### Table")][0])

INFO:text_extract:openpyxl method cannot read xlrd files; using the pandas extractor


legacy .xls with method='openpyxl' -> first line: | 0 | 1 | 2 | 3 | 4 | 5 |  (pandas was used)
with fallback_to_pandas=False     -> ValueError: The openpyxl method cannot read this file type (engine would be 'xlrd')


misnamed .xlsx (named .xls)       -> first table title:

 ### Table 1 — 'Loss Run'!B2:I3


## 13. Try your own file

Set `MY_FILE` (a path, or a name inside `samples/`) and choose the method and options.

In [18]:
MY_FILE = ""                       # e.g. "/Users/me/Downloads/report.xlsx"
METHOD = "openpyxl"                # "pandas" or "openpyxl"
OPTIONS = {}                       # e.g. {"include_comments": True, "detect_general_dates": True}   (openpyxl only)

if MY_FILE:
    for s in h.run("excelExtractText", MY_FILE, method=METHOD, **OPTIONS):
        print(f"##### {s['sheet_name']}\n{s['text'][:4000]}\n")
else:
    print("Set MY_FILE above and re-run this cell.")

Set MY_FILE above and re-run this cell.


---
Reference: [`docs/excel_to_markdown.md`](../docs/excel_to_markdown.md) — options, output examples, the full date-detection rules, limits. The same workbook is used by `tests/test_excel_to_markdown.py`.